# Tests de integración de `persistencia de viajes/trips`

Notebook orientado a verificar el comportamiento end-to-end de OP-06 `write trips`y OP-07 `read trips`,
considerando resultado tabular, resultado operacional y trazabilidad.

In [2]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic" 

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- imports generales,
- imports del módulo,
- helpers de testing reutilizables,
- smoke de imports,
- y configuración básica de display.

Aquí todavía no se prueban escenarios funcionales de `import_trips_from_dataframe()`;
solo se prepara el entorno de trabajo para las secciones siguientes.

### 0.1 Imports generales

Qué prepara: imports base, utilidades de filesystem, PyArrow para inspección parquet y deep copy para no contaminar fixtures.

In [7]:
from pathlib import Path
import copy
import json
import shutil

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

### 0.2 Imports del módulo 

In [3]:
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema
from pylondrina.datasets import TripDataset
from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.validation import ValidationOptions, validate_trips
from pylondrina.errors import ExportError, ValidationError
from pylondrina.io.trips import (
    write_trips,
    read_trips,
    WriteTripsOptions,
    ReadTripsOptions,
)

### 0.3 Import del generador sintético

Qué prepara: acceso al generador para crear fixtures ricas.

In [4]:
from scripts.synthetic_data.base_generator import generate_synthetic_trip_dataframe

### 0.4 Helpers de testing reutilizables

In [5]:
def show_ok(label: str):
    print(f"OK - {label}")


def get_issue_codes(issues):
    return [issue.code for issue in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente {code}. Codes actuales: {codes}"


def clone_tripdataset(trips: TripDataset) -> TripDataset:
    return copy.deepcopy(trips)


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e

### 0.5 Configuración de display

In [8]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

### 0.6 Carpeta visible de integración

Qué prepara: root visible de artefactos de prueba. No usa carpetas temporales ocultas

In [9]:
IT_ROOT = REPO_ROOT / "notebooks" / "testing" / "io_trips" / "tmp_integration"


def reset_it_root() -> Path:
    if IT_ROOT.exists():
        shutil.rmtree(IT_ROOT)
    IT_ROOT.mkdir(parents=True, exist_ok=True)
    return IT_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = IT_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


root = reset_it_root()
print("IT_ROOT =", root.resolve())
show_ok("Sección 0 lista")

IT_ROOT = D:\Memoria\repos\pylondrina\notebooks\testing\io_trips\tmp_integration
OK - Sección 0 lista


## Sección 1. Fixtures ricas reutilizables

### 1.1 Constantes de schema para la fixture rica

Qué prepara: required/base fields y dominios canónicos suficientes para import + validate + persistencia.

In [10]:
REQUIRED_FIELDS_ORDER = [
    "movement_id",
    "user_id",
    "origin_longitude",
    "origin_latitude",
    "destination_longitude",
    "destination_latitude",
    "origin_h3_index",
    "destination_h3_index",
    "origin_time_utc",
    "destination_time_utc",
    "trip_id",
    "movement_seq",
]

REQUIRED_FIELD_DTYPES = {
    "movement_id": "string",
    "user_id": "string",
    "origin_longitude": "float",
    "origin_latitude": "float",
    "destination_longitude": "float",
    "destination_latitude": "float",
    "origin_h3_index": "string",
    "destination_h3_index": "string",
    "origin_time_utc": "datetime",
    "destination_time_utc": "datetime",
    "trip_id": "string",
    "movement_seq": "int",
}

BASE_FIELD_DTYPES = {
    "origin_municipality": "string",
    "destination_municipality": "string",
    "timezone_offset_min": "int",
    "origin_time_local_hhmm": "string",
    "destination_time_local_hhmm": "string",
    "trip_weight": "float",
    "mode_sequence": "string",
    "mode": "categorical",
    "purpose": "categorical",
    "day_type": "categorical",
    "time_period": "categorical",
    "user_gender": "categorical",
    "user_age_group": "categorical",
    "income_quintile": "categorical",
}

CANONICAL_DOMAINS = {
    "mode": [
        "walk", "bicycle", "scooter", "motorcycle", "car",
        "taxi", "ride_hailing", "bus", "metro", "train", "other",
    ],
    "purpose": [
        "home", "work", "education", "shopping", "errand",
        "health", "leisure", "transfer", "other",
    ],
    "day_type": ["weekday", "weekend", "holiday"],
    "time_period": ["night", "morning", "midday", "afternoon", "evening"],
    "user_gender": ["female", "male", "other", "unknown"],
    "user_age_group": ["0-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65-plus", "unknown"],
    "income_quintile": ["1", "2", "3", "4", "5", "unknown"],
}

RICH_BASE_FIELDS = [
    "origin_municipality",
    "destination_municipality",
    "timezone_offset_min",
    "origin_time_local_hhmm",
    "destination_time_local_hhmm",
    "trip_weight",
    "mode_sequence",
    "mode",
    "purpose",
    "day_type",
    "time_period",
    "user_gender",
    "user_age_group",
    "income_quintile",
]

RICH_EXTRA_COLUMNS = [
    "activity_status",
    "education_level",
    "travel_time_bucket",
    "season",
    "fare_payment_type",
    "bike_lane_usage",
    "home_tenure",
]

### 1.2 Builder de schema rica

Qué prepara: un TripSchema suficientemente rico para import y validación de datasets sintéticos realistas.

In [11]:
def make_field(name: str, dtype: str, *, required: bool = False, domain: DomainSpec | None = None) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=None,
        domain=domain,
    )


def make_rich_trip_schema() -> TripSchema:
    fields = {}

    for field_name in REQUIRED_FIELDS_ORDER:
        fields[field_name] = make_field(
            field_name,
            REQUIRED_FIELD_DTYPES[field_name],
            required=True,
        )

    for field_name, dtype_name in BASE_FIELD_DTYPES.items():
        domain = None
        if dtype_name == "categorical":
            domain = DomainSpec(values=CANONICAL_DOMAINS[field_name], extendable=True)
        fields[field_name] = make_field(
            field_name,
            dtype_name,
            required=False,
            domain=domain,
        )

    return TripSchema(
        version="1.1",
        fields=fields,
        required=list(REQUIRED_FIELDS_ORDER),
        semantic_rules=None,
    )

### 1.3 Builder de source dataframe rica

Qué prepara: un dataframe de entrada más rico que los smoke tests, usando generador sintético e incluyendo varias columnas base y extras.

In [12]:
def build_rich_source_dataframe(seed: int = 20260404, filas: int = 180) -> pd.DataFrame:
    df = generate_synthetic_trip_dataframe(
        filas=filas,
        seed=seed,
        duplicate_mode="none",
        tier_temporal="tier_1",
        tier1_datetime_format="utc_string_z",
        coord_format="numeric",
        h3_mode="provided_valid",
        trip_structure="multistage",
        max_movements_per_trip=3,
        base_fields=RICH_BASE_FIELDS,
        extra_value_domains={
            "mode": ["canon"],
            "purpose": ["canon"],
            "day_type": ["canon"],
            "time_period": ["canon"],
            "user_gender": ["canon"],
            "user_age_group": ["canon"],
            "income_quintile": ["canon"],
        },
        extra_columns=RICH_EXTRA_COLUMNS,
        null_ratio={
            "origin_municipality": 0.03,
            "destination_municipality": 0.03,
        },
    )
    return df

### 1.4 Fixtures base de integración

Qué prepara:

- trip_schema_snapshot_small
- tripdataset_canonical_small
- tripdataset_unvalidated_small
- tripdataset_validated_small

Aunque se llamen small, aquí quedan suficientemente ricas para integración.

In [13]:
trip_schema_snapshot_small = make_rich_trip_schema()

source_df_rich = build_rich_source_dataframe(filas=180)

tripdataset_canonical_small, canonical_import_report = import_trips_from_dataframe(
    source_df_rich,
    trip_schema_snapshot_small,
    source_name="synthetic_rich_trips",
    options=ImportOptions(
        keep_extra_fields=True,
        selected_fields=None,
        strict=False,
        strict_domains=False,
        single_stage=False,
        source_timezone=None,
    ),
    provenance={
        "source": {"name": "synthetic_generator", "entity": "trips"},
        "ingestion": {"created_at_utc": "2026-04-04T00:00:00Z"},
        "notes": ["fixture de integración OP-06/OP-07"],
    },
    h3_resolution=8,
)

assert canonical_import_report.ok is True
assert tripdataset_canonical_small.metadata["is_validated"] is False

tripdataset_unvalidated_small = clone_tripdataset(tripdataset_canonical_small)

tripdataset_validated_small = clone_tripdataset(tripdataset_canonical_small)
validated_report_fixture = validate_trips(
    tripdataset_validated_small,
    options=ValidationOptions(
        strict=False,
        validate_domains="full",
    ),
)
assert validated_report_fixture.ok is True
assert tripdataset_validated_small.metadata["is_validated"] is True

print("source_df_rich.shape =", source_df_rich.shape)
print("canonical shape     =", tripdataset_canonical_small.data.shape)
print("validated shape     =", tripdataset_validated_small.data.shape)
show_ok("Sección 1 lista")

source_df_rich.shape = (180, 33)
canonical shape     = (180, 33)
validated shape     = (180, 33)
OK - Sección 1 lista


### 1.5 Helper para materializar artefactos válidos desde la API pública

Qué prepara: setup reusable para tests de lectura.

In [14]:
def write_valid_artifact(case_dir: Path, artifact_name: str = "artifact_bundle") -> tuple[TripDataset, Path, object]:
    trips = clone_tripdataset(tripdataset_validated_small)
    base_path = case_dir / artifact_name

    report = write_trips(
        trips,
        base_path,
        options=WriteTripsOptions(
            mode="error_if_exists",
            require_validated=True,
            storage_format="parquet",
            parquet_compression="snappy",
            normalize_artifact_dir=True,
        ),
    )

    artifact_dir = case_dir / f"{artifact_name}.golondrina"
    assert report.ok is True
    assert artifact_dir.exists()
    return trips, artifact_dir, report

## Sección 2. Integration tests de write_trips() / read_trips()

### Test 1 - write feliz con dataset rico y normalización de directorio

Qué prueba: caso principal correcto de write_trips() sobre una fixture validada y rica.
Verifica:

- escritura formal,
- normalización a .golondrina,
- report,
- summary,
- parameters,
- sidecar,
- evento en metadata,
- y preservación de trips.data.

In [17]:
case_dir = make_case_dir("test_01_write_happy_normalized")

trips = clone_tripdataset(tripdataset_validated_small)
data_before = trips.data.copy(deep=True)
metadata_before = copy.deepcopy(trips.metadata)

base_path = case_dir / "sample_bundle"
expected_artifact_dir = case_dir / "sample_bundle.golondrina"

report = write_trips(
    trips,
    base_path,
    options=WriteTripsOptions(
        mode="overwrite",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
    ),
)

assert report.ok is True
assert expected_artifact_dir.exists()
assert (expected_artifact_dir / "trips.parquet").exists()
assert (expected_artifact_dir / "trips.metadata.json").exists()

# Report observable
assert report.summary["n_rows"] == len(trips.data)
assert Path(report.summary["path"]) == expected_artifact_dir
assert report.summary["storage_format"] == "parquet"
assert report.parameters["normalize_artifact_dir"] is True
assert Path(report.parameters["path"]) == expected_artifact_dir

# Side effects en metadata del dataset
assert "dataset_id" in trips.metadata
assert "artifact_id" in trips.metadata
assert trips.metadata["is_validated"] is True
assert len(trips.metadata["events"]) == len(metadata_before["events"]) + 1
assert trips.metadata["events"][-1]["op"] == "write_trips"
assert trips.metadata["events"][-1]["parameters"] == report.parameters
assert trips.metadata["events"][-1]["summary"] == report.summary

# El dataframe no debe mutar
pd.testing.assert_frame_equal(trips.data, data_before)

# Sidecar consistente
sidecar = json.loads((expected_artifact_dir / "trips.metadata.json").read_text(encoding="utf-8"))
assert sidecar["dataset_type"] == "trips"
assert sidecar["format"] == "golondrina"
assert sidecar["layout_version"] == "1.1"
assert sidecar["files"]["data"] == "trips.parquet"
assert sidecar["files"]["metadata"] == "trips.metadata.json"
assert sidecar["dataset_id"] == trips.metadata["dataset_id"]
assert sidecar["artifact_id"] == trips.metadata["artifact_id"]
assert "schema" in sidecar
assert "schema_effective" in sidecar
assert "metadata" in sidecar
assert "provenance" in sidecar
assert_json_safe(sidecar, "sidecar")

display(report)
display(trips.data)
show_ok("Test 1 - write feliz con normalización de directorio")

OperationReport(ok=True, issues=[], summary={'n_rows': 180, 'files_written': ['trips.parquet', 'trips.metadata.json'], 'path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_01_write_happy_normalized\\sample_bundle.golondrina', 'dataset_id': 'tripds_965208ef7ea64c81a8a5dc1c690260a4', 'artifact_id': 'art_dcf72604-f357-4c38-a76c-499d1471b163', 'dataset_id_status': 'preserved', 'storage_format': 'parquet'}, parameters={'path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_01_write_happy_normalized\\sample_bundle.golondrina', 'mode': 'overwrite', 'require_validated': True, 'storage_format': 'parquet', 'parquet_compression': 'snappy', 'normalize_artifact_dir': True})

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_status,education_level,travel_time_bucket,season,fare_payment_type,bike_lane_usage,home_tenure
0,m00000,u0033,tm_00000,0,-70.566229,-33.587260,-70.572292,-33.585248,2026-03-06 01:58:00+00:00,2026-03-06 02:29:00+00:00,88b2c57355fffff,88b2c57355fffff,San Miguel,Ñuñoa,-180,01:58,02:29,4.005,bus+metro,car,errand,weekday,afternoon,male,35-44,unknown,studying,postgraduate,11-20,winter,card,sometimes,loaned
1,m00001,u0033,tm_00000,1,-70.588286,-33.302660,-70.574418,-33.287252,2026-03-03 06:54:00+00:00,2026-03-03 07:16:00+00:00,88b2c51b6dfffff,88b2c5ce91fffff,La Florida,San Miguel,-180,06:54,07:16,0.869,metro+car,ride_hailing,transfer,holiday,afternoon,unknown,65-plus,2,unemployed,secondary,0-10,winter,integrated_fare,not_applicable,loaned
2,m00002,u0029,tm_00001,0,-70.658943,-33.546320,-70.660997,-33.518745,2026-03-04 02:57:00+00:00,2026-03-04 03:50:00+00:00,88b2c54633fffff,88b2c54601fffff,Quilicura,Vitacura,-180,02:57,03:50,2.360,walk+bicycle,motorcycle,transfer,weekday,night,unknown,15-24,unknown,unemployed,none,11-20,summer,cash,never,other
3,m00003,u0029,tm_00001,1,-70.430100,-33.641724,-70.456117,-33.641027,2026-03-02 19:52:00+00:00,2026-03-02 21:24:00+00:00,88b2c5774bfffff,88b2c57297fffff,San Miguel,Santiago,-180,19:52,21:24,0.731,train,car,transfer,holiday,night,female,15-24,1,homemaker,primary,60+,winter,free_transfer,always,rented
4,m00004,u0029,tm_00001,2,-70.552786,-33.492235,-70.564205,-33.497834,2026-03-01 22:59:00+00:00,2026-03-02 00:50:00+00:00,88b2c50867fffff,88b2c5095bfffff,Recoleta,La Florida,-180,22:59,00:50,3.256,metro+bus+walk,bus,errand,weekday,afternoon,female,45-54,1,studying,none,41-60,normal_period,cash,always,rented
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,m00175,u0030,tm_00089,0,-70.713095,-33.409518,-70.694072,-33.405102,2026-03-06 19:15:00+00:00,2026-03-06 20:49:00+00:00,88b2c5573bfffff,88b2c55713fffff,Estación Central,Quilicura,-180,19:15,20:49,1.151,metro+bicycle+metro,other,health,weekday,evening,unknown,25-34,4,studying,postgraduate,11-20,winter,integrated_fare,not_applicable,other
176,m00176,u0021,tm_00090,0,-70.553847,-33.553322,-70.587342,-33.583055,2026-03-05 19:54:00+00:00,2026-03-05 21:14:00+00:00,88b2c50983fffff,88b2c57347fffff,Recoleta,Puente Alto,-180,19:54,21:14,4.463,walk+bus,car,education,weekend,afternoon,unknown,35-44,5,working,university,41-60,summer,integrated_fare,not_applicable,rented
177,m00177,u0021,tm_00090,1,-70.696619,-33.587977,-70.634727,-33.589844,2026-03-05 07:48:00+00:00,2026-03-05 09:32:00+00:00,88b2c54435fffff,88b2c544ddfffff,Santiago,Independencia,-180,07:48,09:32,0.505,train+bicycle,other,home,weekend,morning,male,15-24,3,homemaker,secondary,21-40,vacation,card,never,owned
178,m00178,u0021,tm_00090,2,-70.722964,-33.410500,-70.684981,-33.441798,2026-03-01 22:27:00+00:00,2026-03-01 23:11:00+00:00,88b2c55735fffff,88b2c5543dfffff,Vitacura,La Florida,-180,22:27,23:11,0.812,bicycle,metro,education,holiday,afternoon,female,65-plus,3,retired,university,0-10,normal_period,cash,not_applicable,loaned


OK - Test 1 - write feliz con normalización de directorio


### Test 2 - write feliz y verificación de dictionary encoding en campos categóricos

Qué prueba: que write_trips() deja evidencia de dictionary encoding en columnas categóricas persistidas, usando la implementación pública.

In [18]:
case_dir = make_case_dir("test_02_write_dictionary_encoding")

trips = clone_tripdataset(tripdataset_validated_small)
base_path = case_dir / "categorical_bundle"

report = write_trips(
    trips,
    base_path,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
    ),
)

artifact_dir = case_dir / "categorical_bundle.golondrina"
parquet_path = artifact_dir / "trips.parquet"

assert report.ok is True
assert parquet_path.exists()

parquet_file = pq.ParquetFile(parquet_path)
try:
    names = parquet_file.schema_arrow.names

    for col_name in ["mode", "purpose", "day_type", "time_period", "user_gender", "user_age_group", "income_quintile"]:
        if col_name in names:
            col_idx = names.index(col_name)
            encodings = {
                str(enc).upper()
                for enc in parquet_file.metadata.row_group(0).column(col_idx).encodings
            }
            assert any("DICTIONARY" in enc for enc in encodings), (
                f"{col_name} no quedó con dictionary encoding: {encodings}"
            )
finally:
    parquet_file.close()

show_ok("Test 2 - dictionary encoding en write_trips")

OK - Test 2 - dictionary encoding en write_trips


### Test 3 - read feliz usando path sin sufijo y schema desde metadata

Qué prueba: dos cosas a la vez:

- fallback amigable a .golondrina en read_trips();
- reconstrucción de schema desde snapshot del sidecar cuando options.schema=None.

In [21]:
case_dir = make_case_dir("test_03_read_happy_fallback_and_metadata_schema")

written_trips, artifact_dir, write_report = write_valid_artifact(case_dir, artifact_name="bundle")

# Leo usando el path sin sufijo para probar el fallback público.
loaded, read_report = read_trips(
    case_dir / "bundle",
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert read_report.ok is True

# Debe haber resuelto el path efectivo con sufijo canónico.
assert Path(read_report.parameters["path"]) == artifact_dir
assert read_report.parameters["schema"]["source"] == "metadata"

# Read siempre deja el dataset no validado.
assert loaded.metadata["is_validated"] is False

# Identidad preservada desde el artefacto persistido.
assert loaded.metadata["dataset_id"] == written_trips.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == written_trips.metadata["artifact_id"]

# Read agrega evento si keep_metadata=True
assert loaded.metadata["events"][-1]["op"] == "read_trips"
assert loaded.metadata["events"][-1]["parameters"] == read_report.parameters
assert loaded.metadata["events"][-1]["summary"] == read_report.summary

# Debe existir el issue informativo de forzar is_validated=False.
assert_issue_present(read_report.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

# Data reconstruida
pd.testing.assert_frame_equal(
    loaded.data.reset_index(drop=True),
    written_trips.data.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

display(loaded.data)
display(read_report.issues)
display(report.summary)
show_ok("Test 3 - read feliz con fallback y schema desde metadata")

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_status,education_level,travel_time_bucket,season,fare_payment_type,bike_lane_usage,home_tenure
0,m00000,u0033,tm_00000,0,-70.566229,-33.587260,-70.572292,-33.585248,2026-03-06 01:58:00+00:00,2026-03-06 02:29:00+00:00,88b2c57355fffff,88b2c57355fffff,San Miguel,Ñuñoa,-180,01:58,02:29,4.005,bus+metro,car,errand,weekday,afternoon,male,35-44,unknown,studying,postgraduate,11-20,winter,card,sometimes,loaned
1,m00001,u0033,tm_00000,1,-70.588286,-33.302660,-70.574418,-33.287252,2026-03-03 06:54:00+00:00,2026-03-03 07:16:00+00:00,88b2c51b6dfffff,88b2c5ce91fffff,La Florida,San Miguel,-180,06:54,07:16,0.869,metro+car,ride_hailing,transfer,holiday,afternoon,unknown,65-plus,2,unemployed,secondary,0-10,winter,integrated_fare,not_applicable,loaned
2,m00002,u0029,tm_00001,0,-70.658943,-33.546320,-70.660997,-33.518745,2026-03-04 02:57:00+00:00,2026-03-04 03:50:00+00:00,88b2c54633fffff,88b2c54601fffff,Quilicura,Vitacura,-180,02:57,03:50,2.360,walk+bicycle,motorcycle,transfer,weekday,night,unknown,15-24,unknown,unemployed,none,11-20,summer,cash,never,other
3,m00003,u0029,tm_00001,1,-70.430100,-33.641724,-70.456117,-33.641027,2026-03-02 19:52:00+00:00,2026-03-02 21:24:00+00:00,88b2c5774bfffff,88b2c57297fffff,San Miguel,Santiago,-180,19:52,21:24,0.731,train,car,transfer,holiday,night,female,15-24,1,homemaker,primary,60+,winter,free_transfer,always,rented
4,m00004,u0029,tm_00001,2,-70.552786,-33.492235,-70.564205,-33.497834,2026-03-01 22:59:00+00:00,2026-03-02 00:50:00+00:00,88b2c50867fffff,88b2c5095bfffff,Recoleta,La Florida,-180,22:59,00:50,3.256,metro+bus+walk,bus,errand,weekday,afternoon,female,45-54,1,studying,none,41-60,normal_period,cash,always,rented
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,m00175,u0030,tm_00089,0,-70.713095,-33.409518,-70.694072,-33.405102,2026-03-06 19:15:00+00:00,2026-03-06 20:49:00+00:00,88b2c5573bfffff,88b2c55713fffff,Estación Central,Quilicura,-180,19:15,20:49,1.151,metro+bicycle+metro,other,health,weekday,evening,unknown,25-34,4,studying,postgraduate,11-20,winter,integrated_fare,not_applicable,other
176,m00176,u0021,tm_00090,0,-70.553847,-33.553322,-70.587342,-33.583055,2026-03-05 19:54:00+00:00,2026-03-05 21:14:00+00:00,88b2c50983fffff,88b2c57347fffff,Recoleta,Puente Alto,-180,19:54,21:14,4.463,walk+bus,car,education,weekend,afternoon,unknown,35-44,5,working,university,41-60,summer,integrated_fare,not_applicable,rented
177,m00177,u0021,tm_00090,1,-70.696619,-33.587977,-70.634727,-33.589844,2026-03-05 07:48:00+00:00,2026-03-05 09:32:00+00:00,88b2c54435fffff,88b2c544ddfffff,Santiago,Independencia,-180,07:48,09:32,0.505,train+bicycle,other,home,weekend,morning,male,15-24,3,homemaker,secondary,21-40,vacation,card,never,owned
178,m00178,u0021,tm_00090,2,-70.722964,-33.410500,-70.684981,-33.441798,2026-03-01 22:27:00+00:00,2026-03-01 23:11:00+00:00,88b2c55735fffff,88b2c5543dfffff,Vitacura,La Florida,-180,22:27,23:11,0.812,bicycle,metro,education,holiday,afternoon,female,65-plus,3,retired,university,0-10,normal_period,cash,not_applicable,loaned


[Issue(level='info', code='READ.METADATA.VALIDATED_FORCED_FALSE', message="Se forzó metadata['is_validated']=False tras la lectura formal del artefacto de trips.", field=None, source_field=None, row_count=None, details={'previous_value': True, 'new_value': False, 'action': 'force_unvalidated'})]

{'n_rows': 180,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_02_write_dictionary_encoding\\categorical_bundle.golondrina',
 'dataset_id': 'tripds_965208ef7ea64c81a8a5dc1c690260a4',
 'artifact_id': 'art_69ba140f-a053-4ac0-92d7-4d1e340cbbde',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

OK - Test 3 - read feliz con fallback y schema desde metadata


### Test 4 - read feliz con schema explícito y precedencia sobre metadata

Qué prueba: que options.schema manda sobre el snapshot persistido y que un mismatch observable queda trazado.

In [22]:
case_dir = make_case_dir("test_04_read_schema_precedence")

written_trips, artifact_dir, _ = write_valid_artifact(case_dir, artifact_name="bundle")

schema_override = copy.deepcopy(trip_schema_snapshot_small)
schema_override.version = "1.1-override"

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=schema_override,
        strict=False,
        keep_metadata=True,
    ),
)

assert read_report.ok is True
assert loaded.schema.version == "1.1-override"
assert read_report.parameters["schema"]["source"] == "options"
assert_issue_present(read_report.issues, "READ.SCHEMA.MISMATCH")

display(read_report)
show_ok("Test 4 - read con schema explícito y precedencia")

OperationReport(ok=True, issues=[Issue(level='warning', code='READ.SCHEMA.MISMATCH', message='El schema provisto por options no coincide con el snapshot persistido en el sidecar; se usará options.schema según precedencia.', field=None, source_field=None, row_count=None, details={'schema_source': 'options', 'schema_mismatch': True, 'version_options': '1.1-override', 'version_metadata': '1.1', 'required_diff': [], 'fields_diff_sample': [], 'fields_diff_total': 0, 'action': 'use_options_schema'}), Issue(level='info', code='READ.METADATA.VALIDATED_FORCED_FALSE', message="Se forzó metadata['is_validated']=False tras la lectura formal del artefacto de trips.", field=None, source_field=None, row_count=None, details={'previous_value': True, 'new_value': False, 'action': 'force_unvalidated'})], summary={'n_rows': 180, 'n_columns': 33, 'path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_04_read_schema_precedence\\bundle.golondrina', 'storage_format': 'par

OK - Test 4 - read con schema explícito y precedencia


### Test 5 - round-trip básico completo

Qué prueba: pipeline público completo write_trips() -> read_trips(), comparando estructura, identidad, schema, schema_effective, metadata y data.

In [23]:
case_dir = make_case_dir("test_05_roundtrip_basic")

trips_original = clone_tripdataset(tripdataset_validated_small)
data_original = trips_original.data.copy(deep=True)
schema_original = copy.deepcopy(trips_original.schema)
schema_effective_original = copy.deepcopy(trips_original.schema_effective)
provenance_original = copy.deepcopy(trips_original.provenance)
field_corr_original = copy.deepcopy(trips_original.field_correspondence)
value_corr_original = copy.deepcopy(trips_original.value_correspondence)

write_report = write_trips(
    trips_original,
    case_dir / "roundtrip_bundle",
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
    ),
)

loaded, read_report = read_trips(
    case_dir / "roundtrip_bundle",
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

# Identidad
assert loaded.metadata["dataset_id"] == trips_original.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == trips_original.metadata["artifact_id"]

# Post-read siempre fuerza False
assert loaded.metadata["is_validated"] is False

# Data
pd.testing.assert_frame_equal(
    loaded.data.reset_index(drop=True),
    data_original.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

# Schema / schema_effective / provenance / correspondencias
assert loaded.schema.to_dict() == schema_original.to_dict()
assert loaded.schema_effective.to_dict() == schema_effective_original.to_dict()
assert loaded.provenance == provenance_original
assert loaded.field_correspondence == field_corr_original
assert loaded.value_correspondence == value_corr_original

# Eventos
ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert "write_trips" in ops_loaded
assert ops_loaded[-1] == "read_trips"

display(trips_original.data)
display(write_report.summary)
display(loaded.data)
display(read_report.summary)
show_ok("Test 5 - round-trip básico completo")

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_status,education_level,travel_time_bucket,season,fare_payment_type,bike_lane_usage,home_tenure
0,m00000,u0033,tm_00000,0,-70.566229,-33.587260,-70.572292,-33.585248,2026-03-06 01:58:00+00:00,2026-03-06 02:29:00+00:00,88b2c57355fffff,88b2c57355fffff,San Miguel,Ñuñoa,-180,01:58,02:29,4.005,bus+metro,car,errand,weekday,afternoon,male,35-44,unknown,studying,postgraduate,11-20,winter,card,sometimes,loaned
1,m00001,u0033,tm_00000,1,-70.588286,-33.302660,-70.574418,-33.287252,2026-03-03 06:54:00+00:00,2026-03-03 07:16:00+00:00,88b2c51b6dfffff,88b2c5ce91fffff,La Florida,San Miguel,-180,06:54,07:16,0.869,metro+car,ride_hailing,transfer,holiday,afternoon,unknown,65-plus,2,unemployed,secondary,0-10,winter,integrated_fare,not_applicable,loaned
2,m00002,u0029,tm_00001,0,-70.658943,-33.546320,-70.660997,-33.518745,2026-03-04 02:57:00+00:00,2026-03-04 03:50:00+00:00,88b2c54633fffff,88b2c54601fffff,Quilicura,Vitacura,-180,02:57,03:50,2.360,walk+bicycle,motorcycle,transfer,weekday,night,unknown,15-24,unknown,unemployed,none,11-20,summer,cash,never,other
3,m00003,u0029,tm_00001,1,-70.430100,-33.641724,-70.456117,-33.641027,2026-03-02 19:52:00+00:00,2026-03-02 21:24:00+00:00,88b2c5774bfffff,88b2c57297fffff,San Miguel,Santiago,-180,19:52,21:24,0.731,train,car,transfer,holiday,night,female,15-24,1,homemaker,primary,60+,winter,free_transfer,always,rented
4,m00004,u0029,tm_00001,2,-70.552786,-33.492235,-70.564205,-33.497834,2026-03-01 22:59:00+00:00,2026-03-02 00:50:00+00:00,88b2c50867fffff,88b2c5095bfffff,Recoleta,La Florida,-180,22:59,00:50,3.256,metro+bus+walk,bus,errand,weekday,afternoon,female,45-54,1,studying,none,41-60,normal_period,cash,always,rented
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,m00175,u0030,tm_00089,0,-70.713095,-33.409518,-70.694072,-33.405102,2026-03-06 19:15:00+00:00,2026-03-06 20:49:00+00:00,88b2c5573bfffff,88b2c55713fffff,Estación Central,Quilicura,-180,19:15,20:49,1.151,metro+bicycle+metro,other,health,weekday,evening,unknown,25-34,4,studying,postgraduate,11-20,winter,integrated_fare,not_applicable,other
176,m00176,u0021,tm_00090,0,-70.553847,-33.553322,-70.587342,-33.583055,2026-03-05 19:54:00+00:00,2026-03-05 21:14:00+00:00,88b2c50983fffff,88b2c57347fffff,Recoleta,Puente Alto,-180,19:54,21:14,4.463,walk+bus,car,education,weekend,afternoon,unknown,35-44,5,working,university,41-60,summer,integrated_fare,not_applicable,rented
177,m00177,u0021,tm_00090,1,-70.696619,-33.587977,-70.634727,-33.589844,2026-03-05 07:48:00+00:00,2026-03-05 09:32:00+00:00,88b2c54435fffff,88b2c544ddfffff,Santiago,Independencia,-180,07:48,09:32,0.505,train+bicycle,other,home,weekend,morning,male,15-24,3,homemaker,secondary,21-40,vacation,card,never,owned
178,m00178,u0021,tm_00090,2,-70.722964,-33.410500,-70.684981,-33.441798,2026-03-01 22:27:00+00:00,2026-03-01 23:11:00+00:00,88b2c55735fffff,88b2c5543dfffff,Vitacura,La Florida,-180,22:27,23:11,0.812,bicycle,metro,education,holiday,afternoon,female,65-plus,3,retired,university,0-10,normal_period,cash,not_applicable,loaned


{'n_rows': 180,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_05_roundtrip_basic\\roundtrip_bundle.golondrina',
 'dataset_id': 'tripds_965208ef7ea64c81a8a5dc1c690260a4',
 'artifact_id': 'art_9cbfa234-97d9-46a3-bf36-979e95823373',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_h3_index,destination_h3_index,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_status,education_level,travel_time_bucket,season,fare_payment_type,bike_lane_usage,home_tenure
0,m00000,u0033,tm_00000,0,-70.566229,-33.587260,-70.572292,-33.585248,2026-03-06 01:58:00+00:00,2026-03-06 02:29:00+00:00,88b2c57355fffff,88b2c57355fffff,San Miguel,Ñuñoa,-180,01:58,02:29,4.005,bus+metro,car,errand,weekday,afternoon,male,35-44,unknown,studying,postgraduate,11-20,winter,card,sometimes,loaned
1,m00001,u0033,tm_00000,1,-70.588286,-33.302660,-70.574418,-33.287252,2026-03-03 06:54:00+00:00,2026-03-03 07:16:00+00:00,88b2c51b6dfffff,88b2c5ce91fffff,La Florida,San Miguel,-180,06:54,07:16,0.869,metro+car,ride_hailing,transfer,holiday,afternoon,unknown,65-plus,2,unemployed,secondary,0-10,winter,integrated_fare,not_applicable,loaned
2,m00002,u0029,tm_00001,0,-70.658943,-33.546320,-70.660997,-33.518745,2026-03-04 02:57:00+00:00,2026-03-04 03:50:00+00:00,88b2c54633fffff,88b2c54601fffff,Quilicura,Vitacura,-180,02:57,03:50,2.360,walk+bicycle,motorcycle,transfer,weekday,night,unknown,15-24,unknown,unemployed,none,11-20,summer,cash,never,other
3,m00003,u0029,tm_00001,1,-70.430100,-33.641724,-70.456117,-33.641027,2026-03-02 19:52:00+00:00,2026-03-02 21:24:00+00:00,88b2c5774bfffff,88b2c57297fffff,San Miguel,Santiago,-180,19:52,21:24,0.731,train,car,transfer,holiday,night,female,15-24,1,homemaker,primary,60+,winter,free_transfer,always,rented
4,m00004,u0029,tm_00001,2,-70.552786,-33.492235,-70.564205,-33.497834,2026-03-01 22:59:00+00:00,2026-03-02 00:50:00+00:00,88b2c50867fffff,88b2c5095bfffff,Recoleta,La Florida,-180,22:59,00:50,3.256,metro+bus+walk,bus,errand,weekday,afternoon,female,45-54,1,studying,none,41-60,normal_period,cash,always,rented
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,m00175,u0030,tm_00089,0,-70.713095,-33.409518,-70.694072,-33.405102,2026-03-06 19:15:00+00:00,2026-03-06 20:49:00+00:00,88b2c5573bfffff,88b2c55713fffff,Estación Central,Quilicura,-180,19:15,20:49,1.151,metro+bicycle+metro,other,health,weekday,evening,unknown,25-34,4,studying,postgraduate,11-20,winter,integrated_fare,not_applicable,other
176,m00176,u0021,tm_00090,0,-70.553847,-33.553322,-70.587342,-33.583055,2026-03-05 19:54:00+00:00,2026-03-05 21:14:00+00:00,88b2c50983fffff,88b2c57347fffff,Recoleta,Puente Alto,-180,19:54,21:14,4.463,walk+bus,car,education,weekend,afternoon,unknown,35-44,5,working,university,41-60,summer,integrated_fare,not_applicable,rented
177,m00177,u0021,tm_00090,1,-70.696619,-33.587977,-70.634727,-33.589844,2026-03-05 07:48:00+00:00,2026-03-05 09:32:00+00:00,88b2c54435fffff,88b2c544ddfffff,Santiago,Independencia,-180,07:48,09:32,0.505,train+bicycle,other,home,weekend,morning,male,15-24,3,homemaker,secondary,21-40,vacation,card,never,owned
178,m00178,u0021,tm_00090,2,-70.722964,-33.410500,-70.684981,-33.441798,2026-03-01 22:27:00+00:00,2026-03-01 23:11:00+00:00,88b2c55735fffff,88b2c5543dfffff,Vitacura,La Florida,-180,22:27,23:11,0.812,bicycle,metro,education,holiday,afternoon,female,65-plus,3,retired,university,0-10,normal_period,cash,not_applicable,loaned


{'n_rows': 180,
 'n_columns': 33,
 'path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_05_roundtrip_basic\\roundtrip_bundle.golondrina',
 'storage_format': 'parquet',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'tripds_965208ef7ea64c81a8a5dc1c690260a4',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_9cbfa234-97d9-46a3-bf36-979e95823373',
 'artifact_id_status': 'loaded'}

OK - Test 5 - round-trip básico completo


### Test 6 - write fatal por precondición de dataset no validado

Qué prueba: política clave de require_validated=True.

In [24]:
case_dir = make_case_dir("test_06_write_fatal_unvalidated")

trips = clone_tripdataset(tripdataset_unvalidated_small)

raised = None
try:
    write_trips(
        trips,
        case_dir / "should_fail_bundle",
        options=WriteTripsOptions(
            mode="error_if_exists",
            require_validated=True,
            storage_format="parquet",
            parquet_compression="snappy",
            normalize_artifact_dir=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValidationError)
assert not (case_dir / "should_fail_bundle.golondrina").exists()

display(raised)
show_ok("Test 6 - write fatal por dataset no validado")

ValidationError(message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', details={'require_validated': True, 'validated_flag': False, 'path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_06_write_fatal_unvalidated\\should_fail_bundle.golondrina', 'action': 'abort'}, issue=Issue(level='error', code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", field=None, source_field=None, row_count=None, details={'require_validated': True, 'validated_flag': False, 'path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_06_write_fatal_unvalidated\\should_fail_bundle.golondrina', 'action': 'abort'}), issues=(Issue(level='error', code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", fiel

OK - Test 6 - write fatal por dataset no validado


### Test 7 - read fatal por layout inválido (parquet sin sidecar)

Qué prueba: sidecar obligatorio en lectura formal.

In [25]:
case_dir = make_case_dir("test_07_read_fatal_missing_sidecar")
artifact_dir = case_dir / "broken_bundle.golondrina"
artifact_dir.mkdir(parents=True, exist_ok=True)

# Setup manual del caso roto: parquet sin sidecar formal.
tripdataset_validated_small.data.to_parquet(
    artifact_dir / "trips.parquet",
    index=False,
    compression="snappy",
    engine="pyarrow",
)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(
            schema=None,
            strict=False,
            keep_metadata=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)

display(raised)
show_ok("Test 7 - read fatal por sidecar faltante")

ExportError(message="El artefacto formal de trips no contiene el sidecar obligatorio 'trips.metadata.json'.", code='READ.LAYOUT.MISSING_SIDECAR', details={'path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_07_read_fatal_missing_sidecar\\broken_bundle.golondrina', 'resolved_path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_07_read_fatal_missing_sidecar\\broken_bundle.golondrina', 'expected_file': 'trips.metadata.json', 'files_present_sample': ['trips.parquet'], 'files_present_total': 1, 'action': 'abort'}, issue=Issue(level='error', code='READ.LAYOUT.MISSING_SIDECAR', message="El artefacto formal de trips no contiene el sidecar obligatorio 'trips.metadata.json'.", field=None, source_field=None, row_count=None, details={'path': 'D:\\Memoria\\repos\\pylondrina\\notebooks\\testing\\io_trips\\tmp_integration\\test_07_read_fatal_missing_sidecar\\broken_bundle.golondrina', 'resolved_path': 'D:\\Memoria\\repos\

OK - Test 7 - read fatal por sidecar faltante


### Test 8 - read degradado con recovery (strict=False)

Qué prueba: caso degradado recuperable:

- schema_effective faltante,
- dataset_id inválido,
- artifact_id inválido,
- strict=False,
- lectura exitosa con warnings y recovery.

In [ ]:
case_dir = make_case_dir("test_08_read_degraded_recovery")

written_trips, artifact_dir, _ = write_valid_artifact(case_dir, artifact_name="bundle")
sidecar_path = artifact_dir / "trips.metadata.json"

payload = json.loads(sidecar_path.read_text(encoding="utf-8"))

# Degradación controlada del sidecar para probar recovery.
payload.pop("schema_effective", None)
payload["dataset_id"] = ""
payload["artifact_id"] = None
payload["metadata"]["dataset_id"] = ""
payload["metadata"]["artifact_id"] = None
payload["metadata"]["is_validated"] = True

sidecar_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert read_report.ok is True

codes = get_issue_codes(read_report.issues)
assert "READ.SCHEMA_EFFECTIVE.DEFAULTED" in codes
assert "READ.METADATA.DATASET_ID_REGENERATED" in codes
assert "READ.METADATA.ARTIFACT_ID_SET_NONE" in codes
assert "READ.METADATA.VALIDATED_FORCED_FALSE" in codes

assert loaded.metadata["is_validated"] is False
assert isinstance(loaded.metadata["dataset_id"], str) and loaded.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] is None

display(read_report.issues)
show_ok("Test 8 - read degradado con recovery")

### Test 9 - política de keep_metadata=False en read

Qué prueba: keep_metadata=False no borra metadata ni provenance; solo evita append del evento read_trips.

In [26]:
case_dir = make_case_dir("test_09_read_keep_metadata_false")

written_trips, artifact_dir, _ = write_valid_artifact(case_dir, artifact_name="bundle")
sidecar = json.loads((artifact_dir / "trips.metadata.json").read_text(encoding="utf-8"))
events_before = copy.deepcopy(sidecar["metadata"]["events"])

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=False,
    ),
)

assert read_report.ok is True
assert loaded.metadata["is_validated"] is False

# No debe appendear evento read_trips.
ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert ops_loaded == [ev["op"] for ev in events_before]
assert "read_trips" not in ops_loaded

# Pero no debe perder metadata ni provenance.
assert "dataset_id" in loaded.metadata
assert loaded.provenance == written_trips.provenance

show_ok("Test 9 - política keep_metadata=False")

OK - Test 9 - política keep_metadata=False


### Test 10 - inspección visible de artefactos creados

Qué prueba: no prueba lógica nueva; deja evidencia e inspección rápida de lo escrito en disco.

In [27]:
print("Contenido de IT_ROOT:")
for p in sorted(IT_ROOT.rglob("*")):
    rel = p.relative_to(IT_ROOT)
    kind = "DIR " if p.is_dir() else "FILE"
    size = "" if p.is_dir() else f" ({p.stat().st_size} bytes)"
    print(f"{kind:4} - {rel}{size}")

Contenido de IT_ROOT:
DIR  - test_01_write_happy_normalized
DIR  - test_01_write_happy_normalized\sample_bundle.golondrina
FILE - test_01_write_happy_normalized\sample_bundle.golondrina\trips.metadata.json (46130 bytes)
FILE - test_01_write_happy_normalized\sample_bundle.golondrina\trips.parquet (41106 bytes)
DIR  - test_02_write_dictionary_encoding
DIR  - test_02_write_dictionary_encoding\categorical_bundle.golondrina
FILE - test_02_write_dictionary_encoding\categorical_bundle.golondrina\trips.metadata.json (46152 bytes)
FILE - test_02_write_dictionary_encoding\categorical_bundle.golondrina\trips.parquet (41106 bytes)
DIR  - test_03_read_happy_fallback_and_metadata_schema
DIR  - test_03_read_happy_fallback_and_metadata_schema\bundle.golondrina
FILE - test_03_read_happy_fallback_and_metadata_schema\bundle.golondrina\trips.metadata.json (46156 bytes)
FILE - test_03_read_happy_fallback_and_metadata_schema\bundle.golondrina\trips.parquet (41106 bytes)
DIR  - test_04_read_schema_precedence